In [1]:

# ============================================================
# JEEVAN LINK - ASSISTANCE DATASET PREPROCESSING AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. FOLDER CONFIGURATION
# ============================================================

BASE_DIR = Path.cwd()

print("ASSISTANCE PREPROCESSING AUDIT")
print("=" * 65)
print("Working directory:", BASE_DIR)

FILES = {
    "connected": "assistance_connected_dataset.csv",
    "final": "assistance_dataset_final.csv",
    "preprocessed": "assistance_dataset_preprocessed_final.csv",
    "label_distribution": "assistance_label_distribution.csv",
    "message_conflicts": "assistance_message_label_conflicts.csv",
    "cleaned": "assistance_model_dataset_final_cleaned.csv",
    "quality_report": "assistance_model_dataset_final_quality_report.txt"
}

# ============================================================
# 2. CHECK WHICH FILES EXIST
# ============================================================

print("\nFILE AVAILABILITY")
print("=" * 65)

for name, filename in FILES.items():
    path = BASE_DIR / filename

    if path.exists():
        print(f"[FOUND] {filename}")
    else:
        print(f"[MISSING] {filename}")

# ============================================================
# 3. LOAD ALL AVAILABLE CSV FILES
# ============================================================

datasets = {}

for name, filename in FILES.items():

    if filename.endswith(".csv") and (BASE_DIR / filename).exists():

        datasets[name] = pd.read_csv(BASE_DIR / filename)

        print(f"\n{name.upper()} DATASET")
        print("-" * 50)
        print("Shape:", datasets[name].shape)
        print("Columns:", datasets[name].columns.tolist())

# ============================================================
# 4. DISPLAY QUALITY REPORT
# ============================================================

report_path = BASE_DIR / FILES["quality_report"]

if report_path.exists():

    print("\nEXISTING DATASET QUALITY REPORT")
    print("=" * 65)

    with open(report_path, "r", encoding="utf-8") as file:
        print(file.read())

else:
    print("\nQuality report file not found.")

# ============================================================
# 5. INSPECT CLEANED DATASET
# ============================================================

if "cleaned" not in datasets:
    raise FileNotFoundError(
        "Cleaned dataset not found. Check the folder and filename."
    )

df = datasets["cleaned"].copy()

print("\nCLEANED DATASET INSPECTION")
print("=" * 65)

print("Shape:", df.shape)
print("\nFirst five records:")
print(df.head())

print("\nData types:")
print(df.dtypes)

# ============================================================
# 6. CHECK MISSING VALUES
# ============================================================

print("\nMISSING VALUE ANALYSIS")
print("=" * 65)

missing = df.isnull().sum()

print(missing[missing > 0])

if missing.sum() == 0:
    print("No missing values found.")

# ============================================================
# 7. CHECK DUPLICATES
# ============================================================

print("\nDUPLICATE ANALYSIS")
print("=" * 65)

print("Duplicate complete rows:", df.duplicated().sum())

if "id" in df.columns:
    print("Duplicate IDs:", df["id"].duplicated().sum())

if "message" in df.columns:
    print(
        "Duplicate messages:",
        df["message"].duplicated().sum()
    )

# ============================================================
# 8. IDENTIFY ASSISTANCE LABELS
# ============================================================

EXPECTED_LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport",
    "clothing",
    "money",
    "missing_people",
    "refugees",
    "death",
    "other_aid",
    "buildings",
    "electricity",
    "tools",
    "hospitals",
    "shops",
    "aid_centers",
    "other_infrastructure",
    "security",
    "military"
]

LABEL_COLUMNS = [
    col for col in EXPECTED_LABELS if col in df.columns
]

print("\nASSISTANCE LABELS FOUND")
print("=" * 65)

print("Number of labels:", len(LABEL_COLUMNS))
print(LABEL_COLUMNS)

# ============================================================
# 9. LABEL DISTRIBUTION
# ============================================================

print("\nLABEL DISTRIBUTION")
print("=" * 65)

label_results = []

for col in LABEL_COLUMNS:

    positive = int((df[col] == 1).sum())
    negative = int((df[col] == 0).sum())

    percentage = (
        positive / len(df) * 100
        if len(df) > 0 else 0
    )

    label_results.append({
        "label": col,
        "positive": positive,
        "negative": negative,
        "positive_percentage": round(percentage, 2)
    })

label_df = pd.DataFrame(label_results)

print(label_df.to_string(index=False))

# ============================================================
# 10. CHECK LABEL VALUES
# ============================================================

print("\nINVALID LABEL VALUE CHECK")
print("=" * 65)

for col in LABEL_COLUMNS:

    values = sorted(df[col].dropna().unique().tolist())

    print(f"{col}: {values}")

    if not set(values).issubset({0, 1}):
        print("WARNING: Non-binary values found!")

# ============================================================
# 11. CHECK RECORDS WITH NO ASSISTANCE LABEL
# ============================================================

if LABEL_COLUMNS:

    zero_label_records = (df[LABEL_COLUMNS].sum(axis=1) == 0)

    multiple_label_records = (df[LABEL_COLUMNS].sum(axis=1) > 1)

    print("\nMULTI-LABEL ANALYSIS")
    print("=" * 65)

    print(
        "Records with zero labels:",
        int(zero_label_records.sum())
    )

    print(
        "Records with multiple labels:",
        int(multiple_label_records.sum())
    )

    print(
        "Records with exactly one label:",
        int((df[LABEL_COLUMNS].sum(axis=1) == 1).sum())
    )

# ============================================================
# 12. CHECK MESSAGE QUALITY
# ============================================================

if "message" in df.columns:

    print("\nMESSAGE QUALITY")
    print("=" * 65)

    message_lengths = df["message"].fillna("").astype(str).str.len()

    print("Empty messages:", (message_lengths == 0).sum())
    print("Average message length:", round(message_lengths.mean(), 2))
    print("Shortest message:", message_lengths.min())
    print("Longest message:", message_lengths.max())

# ============================================================
# 13. CHECK TRAIN / VALIDATION / TEST SPLITS
# ============================================================

if "source_split" in df.columns:

    print("\nDATA SPLIT ANALYSIS")
    print("=" * 65)

    print(df["source_split"].value_counts(dropna=False))

    print("\nLABEL DISTRIBUTION BY SPLIT")

    for split in df["source_split"].dropna().unique():

        split_df = df[
            df["source_split"] == split
        ]

        print(f"\n{split.upper()}")
        print("Records:", len(split_df))

        for col in LABEL_COLUMNS:

            print(
                f"{col}: {int(split_df[col].sum())}"
            )

# ============================================================
# 14. INSPECT MESSAGE-LABEL CONFLICTS
# ============================================================

if "message_conflicts" in datasets:

    conflict_df = datasets["message_conflicts"]

    print("\nMESSAGE-LABEL CONFLICT ANALYSIS")
    print("=" * 65)

    print("Conflict file shape:", conflict_df.shape)
    print("Conflict columns:", conflict_df.columns.tolist())

    print("\nFirst 20 conflict records:")
    print(conflict_df.head(20).to_string(index=False))

# ============================================================
# 15. COMPARE AVAILABLE DATASET VERSIONS
# ============================================================

print("\nDATASET VERSION COMPARISON")
print("=" * 65)

for name, data in datasets.items():

    print(
        f"{name:20} | "
        f"Rows: {len(data):7} | "
        f"Columns: {len(data.columns):3}"
    )

# ============================================================
# 16. SAVE AUDIT RESULTS
# ============================================================

audit_path = BASE_DIR / "assistance_preprocessing_audit.csv"

label_df.to_csv(audit_path, index=False)

print("\nAUDIT COMPLETED")
print("=" * 65)
print("Label audit saved to:", audit_path)
print("No original dataset files were modified.")



ASSISTANCE PREPROCESSING AUDIT
Working directory: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Assitance_Dataset_for_model

FILE AVAILABILITY
[FOUND] assistance_connected_dataset.csv
[FOUND] assistance_dataset_final.csv
[FOUND] assistance_dataset_preprocessed_final.csv
[FOUND] assistance_label_distribution.csv
[FOUND] assistance_message_label_conflicts.csv
[FOUND] assistance_model_dataset_final_cleaned.csv
[FOUND] assistance_model_dataset_final_quality_report.txt


C:\Users\ishit\AppData\Local\Temp\ipykernel_40904\1117516537.py:54: DtypeWarning: Columns (0: sos_id, 1: sos_description, 2: people_trapped, 3: injuries_reported, 4: medical_emergency, 5: water_level, 6: building_condition, 7: rescue_required, 8: food_required, 9: water_required, 10: shelter_required, 11: location_risk, 12: urgency_label, 13: record_source, 14: context_link_status, 15: EventID, 16: Flood Type) have mixed types. Specify dtype option on import or set low_memory=False.
  datasets[name] = pd.read_csv(BASE_DIR / filename)



CONNECTED DATASET
--------------------------------------------------
Shape: (28837, 45)
Columns: ['record_type', 'sos_id', 'sos_description', 'people_affected', 'people_trapped', 'injuries_reported', 'medical_emergency', 'water_level', 'building_condition', 'rescue_required', 'food_required', 'water_required', 'shelter_required', 'location_risk', 'urgency_label', 'record_source', 'context_link_status', 'EventID', 'Peak Flood Level (m)', 'Peak Discharge Q (cumec)', 'Flood Volume (cumec)', 'Event Duration (days)', 'Time to Peak (days)', 'Flood Type', 'rain_T1d', 'rain_T2d', 'rain_T3d', 'rain_T4d', 'rain_T5d', 'rain_T6d', 'rain_T7d', 'rain_T8d', 'rain_T9d', 'rain_T10d', 'id', 'message', 'genre', 'medical_help', 'medical_products', 'search_and_rescue', 'water', 'food', 'shelter', 'transport', 'source_split']

FINAL DATASET
--------------------------------------------------
Shape: (23801, 25)
Columns: ['id', 'message', 'split', 'medical_help', 'medical_products', 'search_and_rescue', 'secu

In [2]:

# ============================================================
# JEEVAN LINK - DETAILED ASSISTANCE LABEL DIAGNOSIS
# READ-ONLY: DOES NOT MODIFY ORIGINAL DATASETS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()

CLEAN_FILE = BASE_DIR / "assistance_model_dataset_final_cleaned.csv"
PREPROCESSED_FILE = BASE_DIR / "assistance_dataset_preprocessed_final.csv"
CONFLICT_FILE = BASE_DIR / "assistance_message_label_conflicts.csv"

LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

clean = pd.read_csv(CLEAN_FILE)
original = pd.read_csv(PREPROCESSED_FILE, low_memory=False)
conflicts = pd.read_csv(CONFLICT_FILE)

# ------------------------------------------------------------
# 1. CHECK INVALID MESSAGES IN FINAL DATASET
# ------------------------------------------------------------

print("=" * 65)
print("1. INVALID MESSAGE CHECK")
print("=" * 65)

invalid_mask = (
    clean["message"].fillna("").astype(str).str.strip().eq("")
    | clean["message"].fillna("").astype(str).str.strip().str.upper().eq("#NAME?")
)

print("Invalid messages:", invalid_mask.sum())

print(clean.loc[invalid_mask, ["id", "message"]].head(30))

# ------------------------------------------------------------
# 2. ZERO-LABEL MESSAGE SAMPLE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("2. ZERO-LABEL MESSAGE SAMPLE")
print("=" * 65)

zero_mask = clean[LABELS].sum(axis=1) == 0
zero_records = clean[zero_mask]

print("Total zero-label records:", len(zero_records))

print("\nRandom sample of 50 zero-label messages:")

zero_sample = zero_records.sample(
    n=min(50, len(zero_records)),
    random_state=42
)

for _, row in zero_sample.iterrows():
    print(f"\nID: {row['id']}")
    print(f"Genre: {row['genre']}")
    print(f"Message: {row['message'][:500]}")

# ------------------------------------------------------------
# 3. CONFLICT RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("3. CONFLICTING MESSAGE GROUPS")
print("=" * 65)

for message, group in conflicts.groupby("message", dropna=False):

    print("\nMESSAGE:", str(message)[:500])
    print("Number of conflicting records:", len(group))

    print(
        group[
            ["id", "genre"] + LABELS + ["source_split"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 4. CHECK LABEL DISTRIBUTION IN ORIGINAL DATA
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("4. ORIGINAL DATASET LABEL CHECK")
print("=" * 65)

available_labels = [
    label for label in LABELS if label in original.columns
]

for label in available_labels:

    print(f"\n{label}")

    print(
        pd.to_numeric(
            original[label],
            errors="coerce"
        ).value_counts(dropna=False).to_dict()
    )

# ------------------------------------------------------------
# 5. CHECK ORIGINAL RECORDS FOR CONFLICT IDS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("5. ORIGINAL RECORDS FOR CONFLICT IDS")
print("=" * 65)

conflict_ids = conflicts["id"].dropna().unique()

if "id" in original.columns:

    original["id"] = pd.to_numeric(
        original["id"],
        errors="coerce"
    )

    for record_id in conflict_ids:

        matching = original[original["id"] == record_id]

        if not matching.empty:

            print(f"\nORIGINAL ID: {record_id}")

            display_columns = [
                col for col in [
                    "id", "message", "genre"
                ] + available_labels
                if col in matching.columns
            ]

            print(
                matching[display_columns].to_string(index=False)
            )

# ------------------------------------------------------------
# 6. CHECK SPLIT DISTRIBUTION FOR EACH LABEL
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("6. LABEL DISTRIBUTION BY SPLIT")
print("=" * 65)

for split, group in clean.groupby("source_split"):

    print(f"\n{split.upper()}")

    for label in LABELS:

        print(
            f"{label}: "
            f"{int(group[label].sum())} positive / "
            f"{len(group)} total"
        )

print("\nDIAGNOSIS COMPLETED.")
print("No original dataset files were modified.")


1. INVALID MESSAGE CHECK
Invalid messages: 0
Empty DataFrame
Columns: [id, message]
Index: []

2. ZERO-LABEL MESSAGE SAMPLE
Total zero-label records: 16540

Random sample of 50 zero-label messages:

ID: 8187
Genre: direct
Message: I would like to know where there is job for university man please?

ID: 2380
Genre: direct
Message: Like we informed tomorrow, I am from Gressier and I am living at Tiboukan. Where we are, we didn't get any aid. I am begging you to take my case in charge. .. .

ID: 27661
Genre: news
Message: Human Rights Watch had earlier [documented](http://reliefweb.int/https://www.hrw.org/news/2013/02/01/mali-malian-army-islamist-groups-executed-prisoners) the summary execution of at least 13 men and enforced disappearance of five others by government soldiers from SÃ©varÃ© and in Konna during January 2013.

ID: 21844
Genre: news
Message: Following disputed presidential elections in December 2001, general strikes and blockades on the main highways crippled Madagascar's eco

In [3]:

# ============================================================
# JEEVAN LINK - FINAL ASSISTANCE DATASET PREPROCESSING
# NO MODEL TRAINING IN THIS NOTEBOOK
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

BASE_DIR = Path.cwd()

SOURCE_FILE = BASE_DIR / "assistance_connected_dataset.csv"

OUTPUT_DIR = BASE_DIR / "Final_Preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Source dataset not found: {SOURCE_FILE}"
    )

LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

REQUIRED_COLUMNS = [
    "id",
    "message",
    "genre",
    "source_split"
] + LABELS

print("=" * 70)
print("JEEVAN LINK - ASSISTANCE DATASET PREPROCESSING")
print("=" * 70)

print("Source:", SOURCE_FILE)
print("Output:", OUTPUT_DIR)


# ============================================================
# 2. LOAD SOURCE DATASET
# ============================================================

df = pd.read_csv(
    SOURCE_FILE,
    low_memory=False
)

print("\nSOURCE DATASET")
print("-" * 70)
print("Shape:", df.shape)

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

df = df[REQUIRED_COLUMNS].copy()


# ============================================================
# 3. NORMALIZE DATA SPLIT NAMES
# ============================================================

df["source_split"] = (
    df["source_split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

split_mapping = {
    "training": "train",
    "train": "train",
    "validation": "validation",
    "valid": "validation",
    "val": "validation",
    "testing": "test",
    "test": "test"
}

df["source_split"] = df["source_split"].map(
    split_mapping
)

unknown_split = df["source_split"].isna()

if unknown_split.any():
    raise ValueError(
        "Unknown or missing source_split values found. "
        "Inspect the source dataset before continuing."
    )

print("\nORIGINAL SPLIT COUNTS")
print(df["source_split"].value_counts())


# ============================================================
# 4. VALIDATE TARGET LABELS
# ============================================================

for col in LABELS:

    numeric_values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    invalid = (
        numeric_values.isna()
        | ~numeric_values.isin([0, 1])
    )

    if invalid.any():
        raise ValueError(
            f"Invalid values found in label: {col}. "
            "Review original annotations before preprocessing."
        )

    df[col] = numeric_values.astype("int8")


# ============================================================
# 5. REMOVE INVALID MESSAGES
# ============================================================

df["message"] = df["message"].fillna("").astype(str)

df["message"] = df["message"].str.strip()

invalid_mask = (
    df["message"].eq("")
    | df["message"].str.upper().eq("#NAME?")
)

invalid_records = df[invalid_mask].copy()

df = df[~invalid_mask].copy()

print("\nINVALID MESSAGE REMOVAL")
print("Removed:", len(invalid_records))


# ============================================================
# 6. KEEP DIRECT SOS-STYLE MESSAGES
# ============================================================

df["genre"] = (
    df["genre"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

direct_df = df[df["genre"] == "direct"].copy()

excluded_genre = df[df["genre"] != "direct"].copy()

print("\nGENRE FILTERING")
print("Direct messages retained:", len(direct_df))
print("Other genre messages excluded:", len(excluded_genre))


# ============================================================
# 7. CREATE NORMALIZED MESSAGE FOR DUPLICATE CHECKING
# ============================================================

def normalize_message(message):

    message = str(message).lower().strip()

    message = re.sub(
        r"\s+",
        " ",
        message
    )

    return message


direct_df["normalized_message"] = (
    direct_df["message"].apply(normalize_message)
)


# ============================================================
# 8. CHECK DUPLICATE IDS WITH DIFFERENT RECORD CONTENT
# ============================================================

direct_df["id"] = direct_df["id"].astype(str).str.strip()

id_message_counts = (
    direct_df.groupby("id")["normalized_message"]
    .nunique()
)

conflicting_ids = id_message_counts[
    id_message_counts > 1
].index

id_conflicts = direct_df[
    direct_df["id"].isin(conflicting_ids)
].copy()

direct_df = direct_df[
    ~direct_df["id"].isin(conflicting_ids)
].copy()

print("\nID CONFLICT CHECK")
print("Conflicting IDs:", len(conflicting_ids))
print("Rows quarantined:", len(id_conflicts))


# ============================================================
# 9. IDENTIFY CONFLICTING LABEL ANNOTATIONS
# ============================================================

# If identical normalized messages have different label vectors,
# we do not guess which annotation is correct.
# All such records are saved for manual review.

label_vector_counts = (
    direct_df
    .groupby("normalized_message")[LABELS]
    .apply(lambda group: len(group.drop_duplicates()))
)

conflicting_messages = label_vector_counts[
    label_vector_counts > 1
].index

label_conflicts = direct_df[
    direct_df["normalized_message"].isin(
        conflicting_messages
    )
].copy()

direct_df = direct_df[
    ~direct_df["normalized_message"].isin(
        conflicting_messages
    )
].copy()

print("\nLABEL CONFLICT HANDLING")
print("Conflicting message groups:", len(conflicting_messages))
print("Rows quarantined:", len(label_conflicts))


# ============================================================
# 10. CHECK CROSS-SPLIT MESSAGE LEAKAGE
# ============================================================

# The same normalized message must not appear in both
# training, validation or test datasets.

split_counts = (
    direct_df.groupby("normalized_message")["source_split"]
    .nunique()
)

cross_split_messages = split_counts[
    split_counts > 1
].index

cross_split_records = direct_df[
    direct_df["normalized_message"].isin(
        cross_split_messages
    )
].copy()

direct_df = direct_df[
    ~direct_df["normalized_message"].isin(
        cross_split_messages
    )
].copy()

print("\nCROSS-SPLIT LEAKAGE CHECK")
print("Cross-split message groups:", len(cross_split_messages))
print("Rows quarantined:", len(cross_split_records))


# ============================================================
# 11. REMOVE EXACT DUPLICATE MESSAGES
# ============================================================

# At this stage, remaining repeated messages have the same
# label vector and belong to the same split.

before_deduplication = len(direct_df)

direct_df = direct_df.drop_duplicates(
    subset=["normalized_message"],
    keep="first"
).copy()

duplicates_removed = (
    before_deduplication - len(direct_df)
)

print("\nDUPLICATE REMOVAL")
print("Duplicate rows removed:", duplicates_removed)


# ============================================================
# 12. FINAL DATASET STRUCTURE
# ============================================================

# Keep only the fields required for future model training.
# normalized_message is used for checking only.

FINAL_COLUMNS = [
    "id",
    "message",
    "genre"
] + LABELS + [
    "source_split"
]

if "source_file" in df.columns:
    FINAL_COLUMNS.append("source_file")

# Restore source_file from original source where available.
if "source_file" in pd.read_csv(
    SOURCE_FILE,
    nrows=0
).columns:

    source_metadata = pd.read_csv(
        SOURCE_FILE,
        usecols=["id", "source_file"],
        low_memory=False
    )

    source_metadata["id"] = (
        source_metadata["id"].astype(str).str.strip()
    )

    direct_df = direct_df.merge(
        source_metadata.drop_duplicates("id"),
        on="id",
        how="left"
    )

    if "source_file" not in FINAL_COLUMNS:
        FINAL_COLUMNS.append("source_file")

final_df = direct_df[FINAL_COLUMNS].copy()


# ============================================================
# 13. KEEP ZERO-LABEL RECORDS
# ============================================================

# Zero-label records are retained as genuine negative examples.
# They are NOT automatically deleted.

zero_label_count = int(
    (final_df[LABELS].sum(axis=1) == 0).sum()
)

multiple_label_count = int(
    (final_df[LABELS].sum(axis=1) > 1).sum()
)

print("\nMULTI-LABEL SUMMARY")
print("Zero-label records retained:", zero_label_count)
print("Multiple-label records retained:", multiple_label_count)


# ============================================================
# 14. FINAL QUALITY VALIDATION
# ============================================================

print("\nFINAL QUALITY CHECKS")
print("=" * 70)

assert final_df["message"].notna().all()
assert final_df["message"].str.strip().ne("").all()

assert not final_df["message"].str.upper().eq("#NAME?").any()

assert not final_df["id"].duplicated().any()

assert not final_df["message"].apply(
    normalize_message
).duplicated().any()

assert final_df[LABELS].isin([0, 1]).all().all()

assert set(final_df["source_split"].unique()).issubset(
    {"train", "validation", "test"}
)

# Check each normalized message occurs in exactly one split.
check_splits = (
    final_df.assign(
        normalized=final_df["message"].apply(normalize_message)
    )
    .groupby("normalized")["source_split"]
    .nunique()
)

assert check_splits.max() <= 1

print("PASS: No empty messages")
print("PASS: No invalid #NAME? messages")
print("PASS: No duplicate IDs")
print("PASS: No duplicate normalized messages")
print("PASS: All labels are binary")
print("PASS: No cross-split message leakage")
print("PASS: Existing split assignments preserved")


# ============================================================
# 15. SAVE FINAL PREPROCESSED DATASET
# ============================================================

FINAL_FILE = (
    OUTPUT_DIR / "assistance_dataset_final_for_training.csv"
)

final_df.to_csv(
    FINAL_FILE,
    index=False
)

# ============================================================
# 16. SAVE MANUAL REVIEW FILES
# ============================================================

label_conflicts.to_csv(
    OUTPUT_DIR / "assistance_label_conflicts_review.csv",
    index=False
)

id_conflicts.to_csv(
    OUTPUT_DIR / "assistance_id_conflicts_review.csv",
    index=False
)

cross_split_records.to_csv(
    OUTPUT_DIR / "assistance_cross_split_duplicates_review.csv",
    index=False
)

invalid_records.to_csv(
    OUTPUT_DIR / "assistance_invalid_messages_removed.csv",
    index=False
)

excluded_genre.to_csv(
    OUTPUT_DIR / "assistance_non_direct_messages_excluded.csv",
    index=False
)


# ============================================================
# 17. SAVE LABEL DISTRIBUTION
# ============================================================

label_distribution = []

for split, group in final_df.groupby("source_split"):

    for label in LABELS:

        label_distribution.append({
            "split": split,
            "label": label,
            "total_records": len(group),
            "positive_records": int(group[label].sum()),
            "negative_records": int(
                (group[label] == 0).sum()
            )
        })

distribution_df = pd.DataFrame(label_distribution)

distribution_df.to_csv(
    OUTPUT_DIR / "assistance_final_label_distribution.csv",
    index=False
)


# ============================================================
# 18. SAVE QUALITY REPORT
# ============================================================

report = {
    "dataset_name": "Jeevan Link Assistance Final Dataset",
    "source_file": SOURCE_FILE.name,
    "total_final_records": int(len(final_df)),
    "total_columns": int(len(final_df.columns)),
    "genre_used": "direct",
    "labels": LABELS,
    "number_of_labels": len(LABELS),
    "training_records": int(
        (final_df["source_split"] == "train").sum()
    ),
    "validation_records": int(
        (final_df["source_split"] == "validation").sum()
    ),
    "testing_records": int(
        (final_df["source_split"] == "test").sum()
    ),
    "zero_label_records_retained": zero_label_count,
    "multiple_label_records": multiple_label_count,
    "invalid_messages_removed": int(len(invalid_records)),
    "non_direct_messages_excluded": int(len(excluded_genre)),
    "id_conflict_rows_quarantined": int(len(id_conflicts)),
    "label_conflict_rows_quarantined": int(len(label_conflicts)),
    "cross_split_rows_quarantined": int(len(cross_split_records)),
    "duplicate_rows_removed": int(duplicates_removed),
    "missing_values": int(final_df.isna().sum().sum()),
    "duplicate_ids": int(final_df["id"].duplicated().sum()),
    "duplicate_messages": int(
        final_df["message"].apply(normalize_message).duplicated().sum()
    )
}

with open(
    OUTPUT_DIR / "assistance_final_quality_report.json",
    "w"
) as file:
    json.dump(report, file, indent=4)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ASSISTANCE DATASET PREPROCESSING COMPLETED")
print("=" * 70)

print("Final dataset shape:", final_df.shape)

print("\nFINAL SPLIT DISTRIBUTION")
print(final_df["source_split"].value_counts())

print("\nFINAL LABEL DISTRIBUTION")

print(
    final_df[LABELS].sum().sort_values(
        ascending=False
    )
)

print("\nSAVED OUTPUT FILES")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print("-", path.name)

print("\nFINAL DATASET LOCATION:")
print(FINAL_FILE.resolve())

print("\nNO MODEL TRAINING WAS PERFORMED.")
print("Preprocessing stage completed.")


JEEVAN LINK - ASSISTANCE DATASET PREPROCESSING
Source: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Assitance_Dataset_for_model\assistance_connected_dataset.csv
Output: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Assitance_Dataset_for_model\Final_Preprocessed

SOURCE DATASET
----------------------------------------------------------------------
Shape: (28837, 45)


ValueError: Unknown or missing source_split values found. Inspect the source dataset before continuing.

In [4]:

import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()

df = pd.read_csv(
    BASE_DIR / "assistance_connected_dataset.csv",
    low_memory=False
)

print("=" * 65)
print("SOURCE SPLIT DIAGNOSIS")
print("=" * 65)

print("\n1. SOURCE SPLIT VALUES INCLUDING MISSING VALUES")

print(
    df["source_split"]
    .value_counts(dropna=False)
)

print("\n2. RECORD TYPE DISTRIBUTION")

if "record_type" in df.columns:
    print(
        df["record_type"]
        .value_counts(dropna=False)
    )

print("\n3. RECORD TYPE VS SOURCE SPLIT")

if "record_type" in df.columns:
    print(
        pd.crosstab(
            df["record_type"],
            df["source_split"],
            dropna=False
        )
    )

print("\n4. RECORDS WITH MISSING SOURCE SPLIT")

missing = df[df["source_split"].isna()]

print("Missing split records:", len(missing))

display(
    missing[
        [
            col for col in [
                "id",
                "record_type",
                "message",
                "genre",
                "record_source",
                "context_link_status",
                "source_split"
            ]
            if col in missing.columns
        ]
    ].head(20)
)

print("\n5. OTHER SOURCE SPLIT VALUES")

print(
    df.loc[
        df["source_split"].notna(),
        "source_split"
    ].astype(str).str.strip().str.lower().value_counts()
)

print("\nDIAGNOSIS FINISHED")


SOURCE SPLIT DIAGNOSIS

1. SOURCE SPLIT VALUES INCLUDING MISSING VALUES
source_split
training      19099
NaN            5016
test           2387
validation     2335
Name: count, dtype: int64

2. RECORD TYPE DISTRIBUTION
record_type
disaster_message    23821
sos                  5016
Name: count, dtype: int64

3. RECORD TYPE VS SOURCE SPLIT
source_split      test  training  validation   NaN
record_type                                       
disaster_message  2387     19099        2335     0
sos                  0         0           0  5016

4. RECORDS WITH MISSING SOURCE SPLIT
Missing split records: 5016


,id,record_type,message,genre,record_source,context_link_status,source_split
0,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
1,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
2,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
3,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
4,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
5,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
6,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
7,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
8,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN
9,NaN,sos,NaN,NaN,synthetic,not_recorded,NaN



5. OTHER SOURCE SPLIT VALUES
source_split
training      19099
test           2387
validation     2335
Name: count, dtype: int64

DIAGNOSIS FINISHED


In [5]:

# ============================================================
# JEEVAN LINK - FINAL ASSISTANCE DATASET PREPROCESSING
# DISASTER MESSAGES ONLY - NO MODEL TRAINING
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path.cwd()

SOURCE_FILE = BASE_DIR / "assistance_connected_dataset.csv"

OUTPUT_DIR = BASE_DIR / "Final_Preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {SOURCE_FILE}")

LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

print("=" * 70)
print("JEEVAN LINK ASSISTANCE PREPROCESSING")
print("=" * 70)

# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(SOURCE_FILE, low_memory=False)

print("Original dataset shape:", df.shape)

# ============================================================
# 3. SEPARATE RECORD TYPES
# ============================================================

df["record_type"] = (
    df["record_type"].fillna("").astype(str).str.strip().str.lower()
)

disaster_df = df[
    df["record_type"] == "disaster_message"
].copy()

sos_df = df[
    df["record_type"] == "sos"
].copy()

print("\nRECORD TYPE SEPARATION")
print("Disaster messages:", len(disaster_df))
print("SOS records:", len(sos_df))

# Save SOS records separately. Do not use them for text training.
sos_df.to_csv(
    OUTPUT_DIR / "assistance_sos_records_separate.csv",
    index=False
)

# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

REQUIRED = [
    "id",
    "message",
    "genre",
    "source_split"
] + LABELS

missing = [col for col in REQUIRED if col not in disaster_df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

disaster_df = disaster_df[REQUIRED].copy()

# ============================================================
# 5. NORMALIZE SPLITS
# ============================================================

split_mapping = {
    "training": "train",
    "train": "train",
    "validation": "validation",
    "valid": "validation",
    "val": "validation",
    "testing": "test",
    "test": "test"
}

disaster_df["source_split"] = (
    disaster_df["source_split"]
    .astype(str).str.strip().str.lower()
    .map(split_mapping)
)

if disaster_df["source_split"].isna().any():
    raise ValueError(
        "Unknown split found in disaster messages. "
        "Inspect source records."
    )

print("\nORIGINAL SPLITS")
print(disaster_df["source_split"].value_counts())

# ============================================================
# 6. VALIDATE LABELS
# ============================================================

for col in LABELS:

    values = pd.to_numeric(
        disaster_df[col], errors="coerce"
    )

    if values.isna().any() or not values.isin([0, 1]).all():
        raise ValueError(f"Invalid target values in {col}")

    disaster_df[col] = values.astype(int)

# ============================================================
# 7. REMOVE INVALID MESSAGES
# ============================================================

disaster_df["message"] = (
    disaster_df["message"].fillna("").astype(str).str.strip()
)

invalid_mask = (
    disaster_df["message"].eq("")
    | disaster_df["message"].str.upper().eq("#NAME?")
)

invalid_df = disaster_df[invalid_mask].copy()

disaster_df = disaster_df[~invalid_mask].copy()

print("\nInvalid messages removed:", len(invalid_df))

# ============================================================
# 8. KEEP DIRECT SOS-STYLE MESSAGES
# ============================================================

disaster_df["genre"] = (
    disaster_df["genre"].fillna("").astype(str).str.strip().str.lower()
)

direct_df = disaster_df[
    disaster_df["genre"] == "direct"
].copy()

non_direct_df = disaster_df[
    disaster_df["genre"] != "direct"
].copy()

print("\nDIRECT MESSAGES:", len(direct_df))
print("NON-DIRECT MESSAGES EXCLUDED:", len(non_direct_df))

# ============================================================
# 9. NORMALIZE MESSAGE FOR DUPLICATE CHECKS
# ============================================================

def normalize_message(message):
    message = str(message).lower().strip()
    message = re.sub(r"\s+", " ", message)
    return message

direct_df["normalized_message"] = (
    direct_df["message"].apply(normalize_message)
)

# ============================================================
# 10. QUARANTINE CONFLICTING IDS
# ============================================================

direct_df["id"] = direct_df["id"].astype(str).str.strip()

id_conflict_counts = (
    direct_df.groupby("id")["normalized_message"].nunique()
)

conflicting_ids = id_conflict_counts[
    id_conflict_counts > 1
].index

id_conflicts = direct_df[
    direct_df["id"].isin(conflicting_ids)
].copy()

direct_df = direct_df[
    ~direct_df["id"].isin(conflicting_ids)
].copy()

# ============================================================
# 11. QUARANTINE CONFLICTING LABEL ANNOTATIONS
# ============================================================

label_vector_counts = (
    direct_df.groupby("normalized_message")[LABELS]
    .apply(lambda group: len(group.drop_duplicates()))
)

conflicting_messages = label_vector_counts[
    label_vector_counts > 1
].index

label_conflicts = direct_df[
    direct_df["normalized_message"].isin(conflicting_messages)
].copy()

direct_df = direct_df[
    ~direct_df["normalized_message"].isin(conflicting_messages)
].copy()

# ============================================================
# 12. CHECK CROSS-SPLIT DUPLICATES
# ============================================================

split_counts = (
    direct_df.groupby("normalized_message")["source_split"].nunique()
)

cross_split_messages = split_counts[
    split_counts > 1
].index

cross_split_records = direct_df[
    direct_df["normalized_message"].isin(cross_split_messages)
].copy()

direct_df = direct_df[
    ~direct_df["normalized_message"].isin(cross_split_messages)
].copy()

# ============================================================
# 13. REMOVE SAME-LABEL DUPLICATE MESSAGES
# ============================================================

before = len(direct_df)

direct_df = direct_df.drop_duplicates(
    subset=["normalized_message"],
    keep="first"
).copy()

duplicates_removed = before - len(direct_df)

# ============================================================
# 14. BUILD FINAL DATASET
# ============================================================

final_columns = [
    "id",
    "message",
    "genre"
] + LABELS + ["source_split"]

final_df = direct_df[final_columns].copy()

# Preserve zero-label records. They are useful negative examples.
zero_labels = int((final_df[LABELS].sum(axis=1) == 0).sum())

# ============================================================
# 15. FINAL VALIDATION
# ============================================================

assert final_df["message"].notna().all()
assert final_df["message"].str.strip().ne("").all()
assert not final_df["message"].str.upper().eq("#NAME?").any()
assert not final_df["id"].duplicated().any()
assert not final_df["message"].apply(normalize_message).duplicated().any()
assert final_df[LABELS].isin([0, 1]).all().all()

# No message should appear in more than one split.
assert (
    final_df.assign(
        normalized=final_df["message"].apply(normalize_message)
    )
    .groupby("normalized")["source_split"]
    .nunique().max() <= 1
)

# ============================================================
# 16. SAVE FINAL DATASET AND REVIEW FILES
# ============================================================

final_df.to_csv(
    OUTPUT_DIR / "assistance_dataset_final_for_training.csv",
    index=False
)

sos_df.to_csv(
    OUTPUT_DIR / "assistance_sos_records_separate.csv",
    index=False
)

invalid_df.to_csv(
    OUTPUT_DIR / "assistance_invalid_messages_removed.csv",
    index=False
)

non_direct_df.to_csv(
    OUTPUT_DIR / "assistance_non_direct_messages_excluded.csv",
    index=False
)

id_conflicts.to_csv(
    OUTPUT_DIR / "assistance_id_conflicts_review.csv",
    index=False
)

label_conflicts.to_csv(
    OUTPUT_DIR / "assistance_label_conflicts_review.csv",
    index=False
)

cross_split_records.to_csv(
    OUTPUT_DIR / "assistance_cross_split_duplicates_review.csv",
    index=False
)

# ============================================================
# 17. LABEL DISTRIBUTION
# ============================================================

distribution = []

for split, group in final_df.groupby("source_split"):

    for label in LABELS:

        distribution.append({
            "split": split,
            "label": label,
            "total_records": len(group),
            "positive": int(group[label].sum()),
            "negative": int((group[label] == 0).sum())
        })

distribution_df = pd.DataFrame(distribution)

distribution_df.to_csv(
    OUTPUT_DIR / "assistance_final_label_distribution.csv",
    index=False
)

# ============================================================
# 18. QUALITY REPORT
# ============================================================

report = {
    "original_records": len(df),
    "disaster_message_records": len(disaster_df) + len(invalid_df),
    "sos_records_separated": len(sos_df),
    "direct_records_before_review": len(
        disaster_df[disaster_df["genre"] == "direct"]
    ),
    "final_records": len(final_df),
    "labels": LABELS,
    "training_records": int((final_df["source_split"] == "train").sum()),
    "validation_records": int((final_df["source_split"] == "validation").sum()),
    "testing_records": int((final_df["source_split"] == "test").sum()),
    "zero_label_records_retained": zero_labels,
    "invalid_messages_removed": len(invalid_df),
    "non_direct_messages_excluded": len(non_direct_df),
    "id_conflict_rows_quarantined": len(id_conflicts),
    "label_conflict_rows_quarantined": len(label_conflicts),
    "cross_split_rows_quarantined": len(cross_split_records),
    "duplicate_rows_removed": duplicates_removed
}

with open(
    OUTPUT_DIR / "assistance_final_quality_report.json",
    "w"
) as f:
    json.dump(report, f, indent=4)

# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ASSISTANCE PREPROCESSING COMPLETED")
print("=" * 70)

print("\nFinal dataset shape:", final_df.shape)

print("\nFinal split distribution:")
print(final_df["source_split"].value_counts())

print("\nFinal label positives:")
print(final_df[LABELS].sum())

print("\nZero-label records retained:", zero_labels)
print("Invalid messages removed:", len(invalid_df))
print("Non-direct messages excluded:", len(non_direct_df))
print("Conflicting label rows quarantined:", len(label_conflicts))
print("Cross-split rows quarantined:", len(cross_split_records))

print("\nFINAL DATASET:")
print(
    (OUTPUT_DIR / "assistance_dataset_final_for_training.csv").resolve()
)

print("\nNo model training performed.")
print("Original source dataset was not modified.")


JEEVAN LINK ASSISTANCE PREPROCESSING
Original dataset shape: (28837, 45)

RECORD TYPE SEPARATION
Disaster messages: 23821
SOS records: 5016

ORIGINAL SPLITS
source_split
train         19099
test           2387
validation     2335
Name: count, dtype: int64

Invalid messages removed: 6

DIRECT MESSAGES: 10765
NON-DIRECT MESSAGES EXCLUDED: 13050

ASSISTANCE PREPROCESSING COMPLETED

Final dataset shape: (10712, 11)

Final split distribution:
source_split
train         8608
test          1063
validation    1041
Name: count, dtype: int64

Final label positives:
food                 1800
water                 832
shelter              1147
medical_help          590
medical_products      470
search_and_rescue     215
transport             210
dtype: int64

Zero-label records retained: 7573
Invalid messages removed: 6
Non-direct messages excluded: 13050
Conflicting label rows quarantined: 8
Cross-split rows quarantined: 16

FINAL DATASET:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Assitan

In [6]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. PATHS
# --------------------------------------------------

BASE_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Assitance_Dataset_for_model"
)

DATA_FILE = BASE_DIR / "Final_Preprocessed" / "assistance_dataset_final_for_training.csv"

REVIEW_DIR = BASE_DIR / "Final_Quality_Review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. LOAD DATASET
# --------------------------------------------------

df = pd.read_csv(DATA_FILE)

labels = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

print("Dataset shape:", df.shape)

# --------------------------------------------------
# 3. IDENTIFY ZERO-LABEL MESSAGES
# --------------------------------------------------

df["total_positive_labels"] = df[labels].sum(axis=1)

zero_label_df = df[df["total_positive_labels"] == 0].copy()

print("\nZero-label records:", len(zero_label_df))
print(
    "Percentage:",
    round(len(zero_label_df) / len(df) * 100, 2),
    "%"
)

# Random sample of 100 zero-label messages
zero_sample = zero_label_df.sample(
    n=min(100, len(zero_label_df)),
    random_state=42
).copy()

zero_sample["review_decision"] = ""
zero_sample["review_notes"] = ""

zero_sample.to_csv(
    REVIEW_DIR / "zero_label_manual_review.csv",
    index=False
)

# --------------------------------------------------
# 4. REVIEW RARE LABELS
# --------------------------------------------------

rare_review_files = []

for label in ["search_and_rescue", "transport"]:

    positive_records = df[df[label] == 1].copy()

    sample = positive_records.sample(
        n=min(30, len(positive_records)),
        random_state=42
    ).copy()

    sample["reviewed_label"] = label
    sample["review_decision"] = ""
    sample["review_notes"] = ""

    output_file = REVIEW_DIR / f"{label}_manual_review.csv"

    sample.to_csv(output_file, index=False)
    rare_review_files.append(output_file)

    print(f"\n{label}:")
    print("Total positive records:", len(positive_records))
    print("Records selected for review:", len(sample))

# --------------------------------------------------
# 5. LABEL DISTRIBUTION BY SPLIT
# --------------------------------------------------

split_distribution = df.groupby("source_split")[labels].sum()

split_distribution.to_csv(
    REVIEW_DIR / "label_distribution_by_split.csv"
)

print("\nLABEL DISTRIBUTION BY SPLIT")
print(split_distribution)

# --------------------------------------------------
# 6. ZERO-LABEL DISTRIBUTION BY SPLIT
# --------------------------------------------------

zero_split = (
    zero_label_df["source_split"]
    .value_counts()
    .rename_axis("source_split")
    .reset_index(name="zero_label_records")
)

zero_split.to_csv(
    REVIEW_DIR / "zero_label_distribution_by_split.csv",
    index=False
)

print("\nZERO-LABEL DISTRIBUTION")
print(zero_split)

# --------------------------------------------------
# 7. SUMMARY
# --------------------------------------------------

summary = {
    "total_records": len(df),
    "zero_label_records": len(zero_label_df),
    "zero_label_percentage": round(
        len(zero_label_df) / len(df) * 100, 2
    ),
    "search_and_rescue_positive": int(df["search_and_rescue"].sum()),
    "transport_positive": int(df["transport"].sum()),
    "missing_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum())
}

pd.DataFrame(
    list(summary.items()),
    columns=["metric", "value"]
).to_csv(
    REVIEW_DIR / "final_quality_summary.csv",
    index=False
)

print("\nQUALITY REVIEW FILES CREATED SUCCESSFULLY")
print("Location:", REVIEW_DIR)
print("\n", summary)


Dataset shape: (10712, 11)

Zero-label records: 7573
Percentage: 70.7 %

search_and_rescue:
Total positive records: 215
Records selected for review: 30

transport:
Total positive records: 210
Records selected for review: 30

LABEL DISTRIBUTION BY SPLIT
              food  water  shelter  medical_help  medical_products  \
source_split                                                         
test           201     97      123            69                46   
train         1426    664      924           452               371   
validation     173     71      100            69                53   

              search_and_rescue  transport  
source_split                                
test                          8         30  
train                       183        154  
validation                   24         26  

ZERO-LABEL DISTRIBUTION
  source_split  zero_label_records
0        train                6124
1   validation                 737
2         test                 712

QUALI